# Bag of word models

Load text

In [27]:
text = ["The quick brown fox jumped over the lazy dog."]

## Sklearn

### Count

In [28]:
from sklearn.feature_extraction.text import CountVectorizer

In [29]:
vectorizer = CountVectorizer()
vectorizer.fit(text)
vectorizer.vocabulary_

{'the': 7,
 'quick': 6,
 'brown': 0,
 'fox': 2,
 'jumped': 3,
 'over': 5,
 'lazy': 4,
 'dog': 1}

In [30]:
vector = vectorizer.transform(text)
vector.toarray()

array([[1, 1, 1, 1, 1, 1, 1, 2]])

### Tfid

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
text = ["The quick brown fox jumped over the lazy dog.",
		"The dog.",
		"The fox"]
vectorizer = TfidfVectorizer()
vectorizer.fit(text)
print(vectorizer.vocabulary_)
print(vectorizer.idf_)

{'the': 7, 'quick': 6, 'brown': 0, 'fox': 2, 'jumped': 3, 'over': 5, 'lazy': 4, 'dog': 1}
[1.69314718 1.28768207 1.28768207 1.69314718 1.69314718 1.69314718
 1.69314718 1.        ]


In [33]:
vector = vectorizer.transform([text[0]])
print(vector.shape)
print(vector.toarray())

(1, 8)
[[0.36388646 0.27674503 0.27674503 0.36388646 0.36388646 0.36388646
  0.36388646 0.42983441]]


### HashingVectorizer

In [34]:
from sklearn.feature_extraction.text import HashingVectorizer

In [35]:
text = ["The quick brown fox jumped over the lazy dog."]
vectorizer = HashingVectorizer(n_features=20)
vector = vectorizer.transform(text)
print(vector.shape)
print(vector.toarray())

(1, 20)
[[ 0.          0.          0.          0.          0.          0.33333333
   0.         -0.33333333  0.33333333  0.          0.          0.33333333
   0.          0.          0.         -0.33333333  0.          0.
  -0.66666667  0.        ]]


## Keras

In [36]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence, one_hot

In [37]:
text = 'The quick brown fox jumped over the lazy dog.'
result = text_to_word_sequence(text)
result

['the', 'quick', 'brown', 'fox', 'jumped', 'over', 'the', 'lazy', 'dog']

In [38]:
words = set(text_to_word_sequence(text))
vocab_size = len(words)
vocab_size

8

In [39]:
result = one_hot(text, round(vocab_size*1.3))
print(result)

[4, 5, 3, 5, 4, 4, 4, 4, 2]


Hash Encoding with hashing_trick

In [40]:
from tensorflow.keras.preprocessing.text import hashing_trick


text = 'The quick brown fox jumped over the lazy dog.'
words = set(text_to_word_sequence(text))
vocab_size = len(words)
print(vocab_size)
result = hashing_trick(text, round(vocab_size*1.3), hash_function='md5')
print(result)

8
[6, 4, 1, 2, 7, 5, 6, 2, 6]


### Tokenizer

In [41]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [42]:
docs = ['Well done!',
		'Good work',
		'Great effort',
		'nice work',
		'Excellent!']

In [43]:
t = Tokenizer()
t.fit_on_texts(docs)
print(t.word_counts)
print(t.document_count)
print(t.word_index)
print(t.word_docs)

OrderedDict([('well', 1), ('done', 1), ('good', 1), ('work', 2), ('great', 1), ('effort', 1), ('nice', 1), ('excellent', 1)])
5
{'work': 1, 'well': 2, 'done': 3, 'good': 4, 'great': 5, 'effort': 6, 'nice': 7, 'excellent': 8}
defaultdict(<class 'int'>, {'well': 1, 'done': 1, 'good': 1, 'work': 2, 'effort': 1, 'great': 1, 'nice': 1, 'excellent': 1})


In [44]:
encoded_docs = t.texts_to_matrix(docs, mode='count')
print(encoded_docs)

[[0. 0. 1. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1.]]


## Torch

### Scratch BoW

In [45]:
import torch
from collections import Counter

In [46]:
texts = [
    "фильм отличный актёрская игра superb",
    "ужасная халтура скучно не смотрите",
    "замечательный сюжет интересно советую",
    "провал года время потрачено зря"
]

In [47]:
word_counts = Counter()
for text in texts:
    word_counts.update(text.lower().split())

In [48]:
vocab = {word: idx+1 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab_size = len(vocab) + 1   # for OOV (out of vocabulary)
vocab

{'фильм': 1,
 'отличный': 2,
 'актёрская': 3,
 'игра': 4,
 'superb': 5,
 'ужасная': 6,
 'халтура': 7,
 'скучно': 8,
 'не': 9,
 'смотрите': 10,
 'замечательный': 11,
 'сюжет': 12,
 'интересно': 13,
 'советую': 14,
 'провал': 15,
 'года': 16,
 'время': 17,
 'потрачено': 18,
 'зря': 19}

In [49]:
def text_to_bow(text, vocab, vocab_size):
    bow = torch.zeros(vocab_size)
    for word in text.lower().split():
        idx = vocab.get(word, 0)
        bow[idx] += 1
    return bow

In [50]:
vectors = torch.stack([text_to_bow(t, vocab, vocab_size) for t in texts])

In [51]:
vectors

tensor([[0., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0.,
         0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.,
         1., 1.]])